# Qwen3.8 × ARC: three-arm exploratory comparison

The Colab cells below prepare a reproducible paired 50-task sample and separate development and selection sets; they do not execute the experiments. The local Mac workflow later in this notebook implements prompt selection, sandboxed agent proposals, and conditional LoRA training. See its checkpoints and the results notebook for actual status; implementation is not proof of an experiment or score. No official evaluation data or comparison answers are loaded by these preparation cells.

For Colab preparation, use a commit containing the supporting code, upload ARC training challenges, mount Drive, and enter the exact commit. The 50 comparison tasks were previously scored; they are not an untouched test.


In [ ]:
REPO_URL = 'https://github.com/morshoto/qwen-meetup-2.git'
REPO_REVISION = ''  # commit containing src/arc/qwen38_rsi_comparison.py
CHALLENGES = '/content/drive/MyDrive/qwen-colab/inputs/arc-agi_training_challenges.json'
RUN_DIR = '/content/drive/MyDrive/qwen-colab/runs/qwen38-rsi-comparison-v1'
SEED = 20261111


In [ ]:
import hashlib, json, os, re, subprocess, sys
from pathlib import Path
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
if not re.fullmatch(r'[0-9a-f]{40}', REPO_REVISION):
    raise ValueError('Set REPO_REVISION to the full commit SHA containing the comparison code.')
repo = Path('/content/qwen38-rsi-comparison')
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', '-q', REPO_URL, str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', '-q', 'origin', REPO_REVISION], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', REPO_REVISION], check=True)
assert subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip() == REPO_REVISION
sys.path.insert(0, str(repo))
from src.arc.qwen38_rsi_comparison import allocate


In [ ]:
raw = Path(CHALLENGES).read_bytes()
proposal = json.loads((repo / 'configs/lt.phase-a.comparison-proposal.json').read_text())
if hashlib.sha256(raw).hexdigest() != proposal['dataset_sha256']:
    raise ValueError('Challenge dataset SHA differs from the recorded Phase A allocation.')
tasks = json.loads(raw)
allocation = allocate(proposal, tasks, SEED)
registration = {'protocol': 'qwen38-rsi-three-arm-preparation-v1',
                'repo_revision': REPO_REVISION, 'challenge_sha256': proposal['dataset_sha256'],
                'allocation': allocation, 'arms': {name: 'not implemented' for name in ('prompt', 'agent', 'weights')},
                'budget_target_hours': 12}
run = Path(RUN_DIR)
run.mkdir(parents=True, exist_ok=True)
path = run / 'registration.json'
if path.exists():
    if json.loads(path.read_text()) != registration:
        raise ValueError('Registration differs; choose a new RUN_DIR.')
else:
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(registration, indent=2, sort_keys=True) + '\n')
    os.replace(tmp, path)
print('Allocation:', allocation['sha256'])
print('Comparison:', len(allocation['evaluation']), '| development:', len(allocation['development']), '| selection:', len(allocation['selection']))
print('Arms:', registration['arms'])
print('No GPU inference or training has run. A 12-hour target is not an enforced budget.')


## After Colab preparation

This is an allocation checkpoint, not a completed Colab experiment. The local Mac workflow below uses its own frozen model and run registration. Do not combine its 4-bit MLX scores with the earlier Colab FP8/vLLM outputs as if they used matching inference settings.


## Local Qwen3.8 run (separate from this Colab preparation)

The local implementation runs development, prompt selection, the matched baseline, and the prompt evaluation using `src/cli/qwen38_local_experiment.py`. The agent runner evaluates model-authored task analyzers inside a tested restricted container. The weight runner checks verified self-generated data and requires a real-model adapter canary before training. Each arm records resumable per-task checkpoints; a failed gate is reported rather than converted into a score.

The local baseline has completed. The prompt selection retained the original prompt because neither proposal improved on the selection set; its evaluation is therefore a baseline repeat, not an improved prompt. Agent and adapter results depend on their actual execution and gates. Run the offline scorer locally as shown below; the comparison tasks are previously evaluated and exploratory.


### Local experiment commands

The cell above is for Colab allocation preparation. The following are **terminal commands on the Apple Silicon Mac**. They start real model work and can take many hours. Keep the Mac awake and preserve the run directory.

```bash
uv venv --python /opt/homebrew/bin/python3.12 artifacts/qwen38-local-env/venv
uv pip install --python artifacts/qwen38-local-env/venv/bin/python 'mlx-vlm[train]==0.6.8' pytest
artifacts/qwen38-local-env/venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('mlx-community/Qwen3.8-27B-4bit', revision='10c35caafbb80f7dc6a7a432cdd11af10a6d4818'))"
```

Set `MODEL_PATH` to the path printed by the last command and use the same run directory on resume. `data/raw/arc-agi_training_challenges.json` must have the registered Phase A hash. No solutions are read during inference.

```bash
MODEL_PATH="$HOME/.cache/huggingface/hub/models--mlx-community--Qwen3.8-27B-4bit/snapshots/10c35caafbb80f7dc6a7a432cdd11af10a6d4818"
RUN_DIR=artifacts/qwen38-local-20261109
PYTHONPATH=. artifacts/qwen38-local-env/venv/bin/python -m src.cli.qwen38_local_smoke --model-path "$MODEL_PATH" --output "$RUN_DIR/smoke.json"
PYTHONPATH=. artifacts/qwen38-local-env/venv/bin/python -m src.cli.qwen38_local_experiment --model-path "$MODEL_PATH" --run-dir "$RUN_DIR"
```

Agent code is run only after the container checks pass. Run the weight arm after the previous process has stopped; its data and real-model canary gates may correctly block training.

```bash
QWEN38_DOCKER_TEST=1 PYTHONPATH=. artifacts/qwen38-local-env/venv/bin/python -m pytest tests/arc/test_qwen38_agent_sandbox.py -q
PYTHONPATH=. artifacts/qwen38-local-env/venv/bin/python -m src.cli.qwen38_local_agent --model-path "$MODEL_PATH" --run-dir "$RUN_DIR"
PYTHONPATH=. artifacts/qwen38-local-env/venv/bin/python -m src.cli.qwen38_local_train --model-path "$MODEL_PATH" --run-dir "$RUN_DIR"
PYTHONPATH=. artifacts/qwen38-local-env/venv/bin/python -m src.cli.qwen38_local_score --run-dir "$RUN_DIR"
```

The scorer reads the local training solutions, **not** during model inference. Do not publish raw task answers or compare the local 4-bit results as though they were identical to the Colab FP8 run.

If the baseline development attempts do not yield enough verified training examples, the weight arm makes up to six additional seeded attempts per development task and checkpoints each. Repeated identical outputs for the same parent task are not counted as separate training examples.
